In [17]:
# 라이브러리 설치
# !pip install konlpy

In [18]:
from konlpy.tag import Okt
okt = Okt()
print(okt.morphs('나는 학교에 간다'))

['나', '는', '학교', '에', '간다']


# 데이터의 분할
- 일반적인 데이터 분할
    - kFold
        - 무작위로 데이터를 폴드화
    - StratifiedKFold
        - 계층화를 유지하면서 폴드화

In [25]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, KFold

data = {
    'document' : ['A', 'B', 'B', 'D', 'E', 'F', 'G', 'H'],
    'label' : [1, 1, 0, 0, 1, 0, 0, 1],
    'id' : ['a', 'a', 'b', 'b', 'c', 'c', 'd', 'd']
}
df = pd.DataFrame(data)
df


,document,label,id
0,A,1,a
1,B,1,a
2,B,0,b
3,D,0,b
4,E,1,c
5,F,0,c
6,G,0,d
7,H,1,d


In [26]:
# 일반적인 KFold
X = df['document']
Y = df['label']
groups = df['id']

k_folds = KFold(n_splits=2, shuffle=True, random_state=42)
s_folds = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)
sg_folds = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)

In [27]:
for x_idx, y_idx in k_folds.split(X, Y):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
2        B      0  b
3        D      0  b
4        E      1  c
6        G      0  d
  document  label id
0        A      1  a
1        B      1  a
5        F      0  c
7        H      1  d


In [28]:
# 계층화 KFold
for x_idx, y_idx in s_folds.split(X, Y):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
1        B      1  a
3        D      0  b
6        G      0  d
7        H      1  d
  document  label id
0        A      1  a
2        B      0  b
4        E      1  c
5        F      0  c


In [29]:
# 계층별 그룹화 KFold
for x_idx, y_idx in sg_folds.split(X, Y, groups):
    print(df.loc[x_idx])
    print(df.loc[y_idx])
    break

  document  label id
4        E      1  c
5        F      0  c
6        G      0  d
7        H      1  d
  document  label id
0        A      1  a
1        B      1  a
2        B      0  b
3        D      0  b


In [31]:
# 네이버 영화 리뷰 (rationg_train.txt)파일
# panad를 이용하여 txt 파일로드
# 파일을 로드하는데 데이터 간의 분리법은 tab으로 이루어져있다.
df = pd.read_csv('../data/ratings_train.txt', sep='\t')
df

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [32]:
# 정보를 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [ ]:
# info()를 통해서 document 컬럼의 결측치가 확인 -> 5개
# 150000개에서 5개의 데이터는 제거 가능
# case1 (isna() + any()) -> 인덱스 조건식으로 조건을 부정하여 사용
df.loc[~df.isna().any(axis=1)]

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
...,...,...,...
149995,6222902,인간이 문제지.. 소는 뭔죄인가..,0
149996,8549745,평점이 너무 낮아서...,1
149997,9311800,이게 뭐요? 한국인은 거들먹거리고 필리핀 혼혈은 착하다?,0
149998,2376369,청춘 영화의 최고봉.방황과 우울했던 날들의 자화상,1


In [36]:
df.dropna(inplace=True)

In [ ]:
# document가 같은 문장이라면 문장을 하나만 두고 나머지는 제거
# 중복 데이터가 존재하는가?
df['document'].value_counts()

document
굿                                      181
good                                    92
최고                                      85
쓰레기                                     79
별로                                      66
                                      ... 
1%라도 기대했던 내가 죄인입니다 죄인입니다....             1
아직도 이 드라마는 내인생의 최고!                      1
패션에 대한 열정! 안나 윈투어!                       1
키이라 나이틀리가 연기하고자 했던건 대체 정신장애일까 틱장애일까      1
흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나        1
Name: count, Length: 146182, dtype: int64

In [42]:
# 중복된 데이터는 제거
# 제거하기 전 데이터의 개수
before = len(df)
# drop_duplicates() : 데이터의 중복을 제거하는 함수
df = df.drop_duplicates('document').reset_index(drop=True)
after = len(df)
print("중복 데이터 제거한 행의 개수 : ", before-after)

중복 데이터 제거한 행의 개수 :  3813


In [43]:
# id 컬럼의 유일한 데이터들의 길이를 확인
print(len(
    df['id'].unique()
    ))

146182


In [44]:
len(df)

146182

In [45]:
# label 컬럼의 데이터의 개수를 확인
df['label'].value_counts()

label
0    73342
1    72840
Name: count, dtype: int64

In [59]:
# train, validation, test 데이터셋으로 8:1:1 정도의 비율로 데이터를 분할
# label의 비율에 맞게 데이터를 나눠준다.
from sklearn.model_selection import train_test_split
# sklearn에는 3개의 데이터셋으로 나눠주는 함수는 존재x
# train_test_split를 2번 사용
X = df['document'].values
Y = df['label'].values
# test 데이터셋을 10%로 먼저 나눠준다.
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=42, test_size=0.1, stratify=Y)
# validation 데이터셋을 11% 정도로 나눠준다. (label 데이터의 비율에 맞게)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train, Y_train, random_state=42, test_size=0.11, stratify=Y_train)

In [ ]:
# 데이터의 분할 정도를 확인
print(round(len(X_train)/ len(X) * 100, 0), round(len(X_test)/ len(X)*100,0), round(len(X_valid)/ len(X)*100), 0)

80.0 10.0 10 0


In [82]:
print(pd.Series(Y_train).value_counts('0'))
print(pd.Series(Y_test).value_counts('0'))
print(pd.Series(Y_valid).value_counts('0'))

0    0.501712
1    0.498288
Name: proportion, dtype: float64
0    0.501744
1    0.498256
Name: proportion, dtype: float64
0    0.501727
1    0.498273
Name: proportion, dtype: float64


In [85]:
# 계층 폴드화 -> 학습 데이터를 분할/학습하여 일반적인 성능을 나타내는 폴드화, 하이퍼 파라미터 탐색과 같이 사용
folds = []
# enumerate() -> 리스트에서 위치의 값으로 데이터를 나눠서 되돌려준다.
# s_folds.split(X_train, Y_train) -> 결과 값이 (tr_idx, va_idx) 형태로 나온다.
for fold, (tr_idx, va_idx) in enumerate(s_folds.split(X_train, Y_train)):
    folds.append(
        {
            'fold' : fold,
            'tr_idx' : tr_idx,
            'va_idx' : va_idx
        }
    )

In [88]:
# folds에서 첫번째 데이터에서 tr_idx 가지고 Y_train의 0, 1의 비율을 확인
test_idx = folds[0]['tr_idx']
pd.Series(Y_train[test_idx]).value_counts()

0    29373
1    29172
Name: count, dtype: int64

# 단어의 토큰화
- 문장을 단어로 잘라준다.
    - 공백을 기준으로 문자를 자른다.
        - 영문에서는 사용 가능, 한글에서는 의미가 손실되는 경우가 발생
    - 형태소를 사용하여 문자를 나눠준다.
        - 국어 사전을 로드하여 단어별로 나눠준다.
        

In [90]:
# 공백을 기반으로 데이터를 나눈다.
text = '나는 학교에 간다'

In [91]:
tokens = text.split()
print(tokens)

['나는', '학교에', '간다']


In [93]:
text2 = 'Hello World'
tokens2 = text2.split()
print(tokens2)

['Hello', 'World']


In [ ]:
from konlpy.tag import Okt
okt = Okt()

print(okt.morphs(text))     # 단어별로 나눠주는 함수

print(okt.pos(text))        # 단어와 단어의 종류를 출력
text_pos = okt.pos(text)

['나', '는', '학교', '에', '간다']
[('나', 'Noun'), ('는', 'Josa'), ('학교', 'Noun'), ('에', 'Josa'), ('간다', 'Noun')]


In [96]:
# 반복문을 이용하여 각원소들을 대입하여 실행
for t in text_pos:
    # print(t)
    # t -> tuple -> 두번째 깊이 'Josa'가 아니라면
    if t[1] != 'Josa':
        print(t[0])

나
학교
간다


In [98]:
okt.nouns(text)

['나', '학교', '간다']

In [102]:
# Okt 로드한 데이터를 이용하여 Okt 형태소 분석
for idx, t in enumerate(X_train):
    if idx == 5:
        break
    _pos = okt.morphs(t)
    print(_pos)

['이런', '감동', '...', '삶', '의', '희망이', '된다']
['초등학교', '때', '이', '거', '200', '번', '받음', '..', '정말', '짱']
['아이엠', '옴티머', '스프', '라임']
['나', '만', '재밋', '게', '봤나']
['최고', '의', '영화', '평점', '1', '점주', '는', '초딩', '들', '은', '대체', '뭐', '냐', '?', '요새', '한국', '영화', '들', '보다', '훨', '낫다', '.', '영화', '보고', '10년', '넘게', '기억', '에', '남았던', '명작', '이다', '.']
